### Setup

In [1]:
# setup
import os
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path("../..").resolve()
load_dotenv(ROOT / ".env")
DB_PATH = ROOT / os.getenv("DUCKDB_PATH", "data/duckDB/primer_impacto.duckdb")
STAGING = "staging"
MARTS = "marts"

In [2]:
# functions
%run -i utils/inspect_functions.ipynb

### Levels

In [3]:
run_sql("""
    select s.schema_name as level,
           count(t.table_name) as tables,
           count(t.table_name) filter (where t.table_type = 'VIEW') as views
    from information_schema.schemata as s
    left join information_schema.tables as t on t.table_schema = s.schema_name
    where s.catalog_name = current_database()
      and s.schema_name not in ('information_schema', 'pg_catalog')
    group by s.schema_name
    order by case s.schema_name when 'raw' then 1 when 'staging' then 2 when 'marts' then 3 else 4 end
""")

,level,tables,views
0,raw,10,0
1,staging,10,10
2,marts,15,0
3,main,0,0
